## Data Cleaning 

In [27]:
import pandas as pd
import re

years = list(range(2015, 2026))

# import CAD data
cad_list = []
for year in years:
    df = pd.read_csv(f"Eugene_CAD_data_noloc/EugeneCAD{year}noloc.csv", low_memory=False)
    cad_list.append(df)

cad = pd.concat(cad_list, ignore_index=True)
cad.head()

,yr,agency,service,inci_id,calltime,case_id,callsource,nature,closecode,closed_as,secs_to_disp,secs_to_arrv,secs_to_close,disp,arrv,priority,primeunit,units_dispd,units_arrived,month
0,2015,EPD,LAW,15000001,2015-01-01 00:00:00.000,NaN,SELF,PERSON STOP,ASST,ASSISTED,0.0,0.0,217,1,1,6,_5E48,1,1,NaN
1,2015,EPD,LAW,15000002,2015-01-01 00:00:44.000,NaN,SELF,FIGHT,RSLV,RESOLVED,0.0,0.0,2114,1,1,P,_3F65,4,2,NaN
2,2015,EPD,LAW,15000003,2015-01-01 00:01:05.000,NaN,PHONE,CHECK WELFARE,ASST,ASSISTED,708.0,1094.0,1538,1,1,5,_3J79,1,1,NaN
3,2015,EPD,LAW,15000007,2015-01-01 00:03:16.000,NaN,W911,SHOTS FIRED,PCHK,PATROL CHECK,277.0,576.0,891,1,1,3,_5E48,2,2,NaN
4,2015,EPD,LAW,15000010,2015-01-01 00:03:34.000,NaN,SELF,ILLEGAL FIREWORKS,ADVI,ADVISED,0.0,0.0,150,1,1,5,_5K97,1,1,NaN


In [28]:
# function to group raw call descriptions into broader call_type categories
def recode_call_type(x):
    x = str(x).lower()

    if any(word in x for word in ["suicid", "mental", "disorganized", "agitation", "harm/risk", "social/mental"]):
        return "Mental Health"

    elif any(word in x for word in ["check welfare", "welfare"]):
        return "Welfare Check"

    elif any(word in x for word in ["dispute", "disorderly"]):
        return "Dispute"

    elif any(word in x for word in ["assist", "public"]):
        return "Public Assistance"

    else:
        return "Other"

In [29]:
# clean cad
cad_clean = cad[["calltime", "nature", "primeunit"]].copy()

# rename
cad_clean = cad_clean.rename(columns={
    "calltime": "call_time",
    "nature": "raw_call_type"
})

# fix values
cad_clean["primeunit"] = cad_clean["primeunit"].astype(str).str.strip()
cad_clean["call_time"] = pd.to_datetime(cad_clean["call_time"], errors="coerce")

# grouped call type
cad_clean["call_type"] = cad_clean["raw_call_type"].apply(recode_call_type)

# identify response
cahoots_identifiers = r"1J77|3J79|3J78|3J77|4J79|3J81|3J76|2J28|2J29|CAHOOT|CAHOT|CAHO"

cad_clean["response"] = cad_clean["primeunit"].apply(
    lambda x:
        "Unknown" if pd.isna(x) or str(x).strip() == ""
        else "CAHOOTS" if re.search(cahoots_identifiers, str(x))
        else "EPD"
)

cad_clean = cad_clean[cad_clean["response"] != "Unknown"]

# final columns
cad_clean = cad_clean[["call_time", "raw_call_type", "call_type", "response"]]

In [30]:
# import MCSLC data
mcslc = pd.read_excel("MCSLC.xlsx")
mcslc.head()

,ID,End Point of Dispatch,City,Dispatch Request Date & Time,Dispatch Date & Time,Arrival on Scene Date & Time,Engagement with Client Date & Time,MCIT Departure Date & Time,Reason for Dispatch #1,Reason for Dispatch #2,Reason for Dispatch #3,Reason for Dispatch #4,Reason for Dispatch #5,Disposition,Minutes: Request → Dispatch,Minutes: Dispatch → Arrival,Minutes: Arrival → Engagement,Minutes: Arrival → Departure
0,12236,Cancelled,Eugene,2025-05-12 13:29:00,2025-05-12 13:34:00,2025-05-12 00:00:00,2025-05-12 00:00:00,2025-05-12 14:03:00,Harm/risk of harm to self/others/property,Disorganized behavior,Agitation or disruptive behavior,NaN,NaN,Arrest,5.0,NaN,0.0,NaN
1,12574,Cancelled,Unknown,2025-12-05 17:41:00,2025-12-05 17:47:00,2025-12-05 18:06:00,NaT,2025-12-05 18:12:00,Agitation or disruptive behavior,NaN,NaN,NaN,NaN,Arrest,6.0,19.0,NaN,NaN
2,12888,Cancelled,Eugene,2025-07-25 21:10:00,2025-07-25 21:14:00,2025-07-25 21:28:00,2025-07-25 21:30:00,2025-07-25 21:36:00,Agitation or disruptive behavior,NaN,NaN,NaN,NaN,Arrest,4.0,14.0,2.0,8.0
3,10504,Engaged Client,Eugene,2025-03-29 22:29:00,2025-03-29 22:43:00,2025-03-29 23:21:00,2025-03-29 23:28:00,2025-03-29 23:58:00,Harm/risk of harm to self/others/property,Disorganized behavior,Agitation or disruptive behavior,NaN,NaN,Arrest,14.0,38.0,7.0,37.0
4,12220,Engaged Client,Eugene,2025-04-26 18:52:00,2025-04-26 18:54:00,2025-04-26 19:05:00,2025-04-26 19:07:00,2025-04-26 19:32:00,Disorganized behavior,Agitation or disruptive behavior,NaN,NaN,NaN,Arrest,2.0,11.0,2.0,27.0


In [31]:
# clean MCSLC
mcslc_clean = mcslc[mcslc["City"] == "Eugene"].copy()

mcslc_clean["call_time"] = pd.to_datetime(mcslc_clean["Dispatch Request Date & Time"], errors="coerce")

mcslc_clean["raw_call_type"] = mcslc_clean["Reason for Dispatch #1"]

mcslc_clean["call_type"] = mcslc_clean["raw_call_type"].apply(recode_call_type)

mcslc_clean["response"] = "MCSLC"

mcslc_clean = mcslc_clean[["call_time", "raw_call_type", "call_type", "response"]]

In [32]:
combined = pd.concat([cad_clean, mcslc_clean], ignore_index=True)

combined["year"] = combined["call_time"].dt.year
combined["month"] = combined["call_time"].dt.month

shutdown_date = pd.to_datetime("2025-04-08")

combined["period"] = np.where(combined["call_time"] < shutdown_date, "pre","post")

combined.to_csv("combined_calls_data.csv", index=False)

In [33]:
combined.head()

,call_time,raw_call_type,call_type,response,year,month,period
0,2015-01-01 00:00:00,PERSON STOP,Other,EPD,2015.0,1.0,pre
1,2015-01-01 00:00:44,FIGHT,Other,EPD,2015.0,1.0,pre
2,2015-01-01 00:01:05,CHECK WELFARE,Welfare Check,CAHOOTS,2015.0,1.0,pre
3,2015-01-01 00:03:16,SHOTS FIRED,Other,EPD,2015.0,1.0,pre
4,2015-01-01 00:03:34,ILLEGAL FIREWORKS,Other,EPD,2015.0,1.0,pre


In [34]:
# save to csv
combined.to_csv("combined_calls_data.csv", index=False)

In [35]:
#import Springfield
springfield_raw = pd.read_excel("2015-2025 SPD Calls for Service.xlsx", sheet_name=None)


In [36]:
springfield=springfield_raw
dfs = []

for year, df in springfield.items():
    df["year"] = year
    dfs.append(df)

springfield_calls = pd.concat(dfs, ignore_index=True)

springfield_calls.columns = (springfield_calls.columns.str.lower().str.replace(" ", "_"))
springfield_calls.head()

,incident_number,initial_call_type,final_call_type,responding_agency,primary_responding_unit,call_creation_time,first_dispatched_time,first_arrival_time,clear_time,priority,call_creation_mechanism,year
0,15000074,ALMAUD,AUDIBLE ALARM,SPD,1S18,2015-01-01 01:16:21,2015-01-01 01:18:28,2015-01-01 01:21:13,2015-01-01 01:32:08,3,PHONE,2015
1,15000083,TRFSTP,DWS,SPD,3D2,2015-01-01 01:22:29,2015-01-01 01:22:29,2015-01-01 01:22:29,2015-01-01 01:34:57,4,SELF,2015
2,15000167,MVAUNK,MOTOR VEH ACC UNKNOWN INJ,SPD,,2015-01-01 03:15:28,NaT,NaT,NaT,3,E911,2015
3,15000216,TRFSTP,DWS,SPD,1S22,2015-01-01 05:27:24,2015-01-01 05:27:24,2015-01-01 05:27:24,2015-01-01 05:38:53,4,SELF,2015
4,15000222,SUSPVE,SUSPICIOUS VEHICLE,SPD,1S11,2015-01-01 05:52:53,2015-01-01 05:53:57,2015-01-01 05:56:21,2015-01-01 06:04:04,3,PHONE,2015


In [37]:
springfield_calls = springfield_calls[["call_creation_time", "final_call_type"]].copy()

springfield_calls = springfield_calls.rename(columns={"call_creation_time": "call_time","final_call_type": "raw_call_type"})

springfield_calls["call_time"] = pd.to_datetime(springfield_calls["call_time"],errors="coerce")

springfield_calls["call_type"] = springfield_calls["raw_call_type"].apply(recode_call_type)

springfield_calls["response"] = "SPD"
springfield_calls["city"] = "Springfield"

springfield_calls = springfield_calls[
    ["call_time", "raw_call_type", "call_type", "response", "city"]
]

In [38]:
springfield_calls.head()

,call_time,raw_call_type,call_type,response,city
0,2015-01-01 01:16:21,AUDIBLE ALARM,Other,SPD,Springfield
1,2015-01-01 01:22:29,DWS,Other,SPD,Springfield
2,2015-01-01 03:15:28,MOTOR VEH ACC UNKNOWN INJ,Other,SPD,Springfield
3,2015-01-01 05:27:24,DWS,Other,SPD,Springfield
4,2015-01-01 05:52:53,SUSPICIOUS VEHICLE,Other,SPD,Springfield


In [39]:
# combine Eugene CAD with Springfield
cad_clean["city"] = "Eugene"
combined_springfield = pd.concat([cad_clean, springfield_calls], ignore_index=True)

# add time variables
combined_springfield["year"] = combined_springfield["call_time"].dt.year
combined_springfield["month"] = combined_springfield["call_time"].dt.month

# add period variable
shutdown_date = pd.Timestamp("2025-04-08")

combined_springfield["period"] = np.where(combined_springfield["call_time"] < shutdown_date, "pre", "post")

combined_springfield.head()

,call_time,raw_call_type,call_type,response,city,year,month,period
0,2015-01-01 00:00:00,PERSON STOP,Other,EPD,Eugene,2015,1,pre
1,2015-01-01 00:00:44,FIGHT,Other,EPD,Eugene,2015,1,pre
2,2015-01-01 00:01:05,CHECK WELFARE,Welfare Check,CAHOOTS,Eugene,2015,1,pre
3,2015-01-01 00:03:16,SHOTS FIRED,Other,EPD,Eugene,2015,1,pre
4,2015-01-01 00:03:34,ILLEGAL FIREWORKS,Other,EPD,Eugene,2015,1,pre


In [40]:
# save Eugene + Springfield
combined_springfield.to_csv("combined_springfield_data.csv", index=False)
combined.to_csv("combined_calls_data.csv", index=False)